In [37]:
from computegraph import types as cgt
import computegraph as cg

import polars as pl
import numpy as np

from jax import numpy as jnp, Array

from summer3.polarized.properties import Property, PropertyTable, LazyExpr
from summer3.polarized.categories import CategoryGroup, CategoryData
from summer3.polarized.flows import FlowSpec, source, dest
from summer3.polarized.expanding import scalar_to_expanding, catdata_to_expanding
from summer3 import polarized as sp
from summer3.managed import ManagedArray, ManagedIndex

pl.Config.set_tbl_rows(64)
pl.Config.set_tbl_width_chars(None)

polars.config.Config

In [38]:
age = Property("age", ["infant", "child", "adult", "older"])
state = Property("state", ["S", "I", "R"])
severity = Property("severity", ["mild", "severe"])

pt = (
    PropertyTable.from_property(state)
    .stratify(severity, state == "I")
    .stratify(age, state)
)

In [39]:
list(pt.uname_prop_map.inv)[0] is state

True

In [40]:
class FlowMap:
    pass


class ExitMap(FlowMap):
    def __init__(self, source_query: LazyExpr):
        self.source_query = source_query


class EntryMap(FlowMap):
    def __init__(self, dest_query: LazyExpr):
        self.dest_query = dest_query


class TransitionMap(FlowMap):
    def __init__(
        self,
        source_query: LazyExpr | CategoryGroup,
        dest_query: LazyExpr | CategoryGroup,
    ):
        self.source_query = source_query
        self.dest_query = dest_query


# +++
# We need to specify absolute/proportional somewhere - presumably on Flow class constructor
# Which always applies to source compartments, so entry flows are always absolute
# Notes:
# Need to provide actualize_flow method (easy enough - get the indices from the model PT and use expanding ops
# to perform adjustments and final multiply
#


class Flow:
    def __init__(
        self,
        name: str,
        fmap: FlowMap,
        param: float | CategoryData | cg.types.GraphObject,
        model,
    ):
        self.name = name
        self.fmap = fmap
        self.base_param = param
        self.model = model
        self.adjustments = []
        # self.meta = {}  +++ We probably will need this for advanced flow queries later - but let's not
        # get ahead of ourselves...

    def adjust(self, adjustment: CategoryData):
        self.adjustments.append(adjustment)

In [41]:
class CompartmentalModel:
    def __init__(self, base_property):
        self._pt = PropertyTable.from_property(base_property)
        self._flows = []

    # +++
    # Consider whether we need pop_split as an argument here - and a possible Stratification class
    def stratify(self, prop: Property, query: LazyExpr):
        """Stratify the scurrent model with a Property; the subset to be stratified
        is specified via query

        Args:
            prop: Property
            query: LazyExpr
        """
        strat_pt = self._pt.stratify(prop, query)
        self._pt = strat_pt

    def add_flow(
        self,
        name: str,
        fmap: FlowMap,
        param: float | CategoryData | cg.types.GraphObject,
    ):
        f = Flow(name, fmap, param, self)
        self._flows.append(f)
        return f

In [42]:
m = CompartmentalModel(state)
f = m.add_flow(
    "infection",
    TransitionMap(state == "S", state == "I"),
    cg.types.Variable("cr", "parameters"),
)
m.stratify(age, ~state.is_null())

In [43]:
m._pt.filter((age == "infant") & (state == "S"))

state_0,age_0,index
str,str,i64
"""S""","""infant""",0


In [44]:
FlowSpec(f.fmap.source_query, f.fmap.dest_query, m._pt).get_flow_pt()

state_0_source,age_0_source,index_source,state_0_dest,age_0_dest,index_dest,index
str,str,i64,str,str,i64,i64
"""S""","""infant""",0,"""I""","""infant""",4,0
"""S""","""child""",1,"""I""","""child""",5,1
"""S""","""adult""",2,"""I""","""adult""",6,2
"""S""","""older""",3,"""I""","""older""",7,3
